In [86]:
!kaggle datasets download -d  aibuzz/covid-toy-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/aibuzz/covid-toy-dataset
License(s): unknown
100%|██████████████████████████████████████████| 787/787 [00:00<00:00, 3.40MB/s]



| Function | Syntax | Joins Along | New Axis Created? | Shape Requirement | Common Use |
|----------|----------|----------|----------|----------|----------|
| `np.concatenate()` | `np.concatenate((a, b), axis=...)` | User specifies (`axis=0` or `1`) | ❌ No | Arrays must match on all axes except the joining axis | General-purpose joining |
| `np.hstack()` | `np.hstack((a, b))` | Horizontal (`axis=1` for 2D) | ❌ No | Same number of rows | Combine feature columns |
| `np.vstack()` | `np.vstack((a, b))` | Vertical (`axis=0`) | ❌ No | Same number of columns | Add rows |
| `np.column_stack()` | `np.column_stack((a, b))` | As columns | ❌ No | Same length | Create feature matrix from 1D arrays |
| `np.stack()` | `np.stack((a, b), axis=...)` | New axis | ✅ Yes | Arrays must have identical shapes | Create higher-dimensional arrays/tensors |

### Parameters

| Parameter | Meaning |
|------------|------------|
| `a, b, ...` | Arrays to combine |
| `axis=0` | Join vertically (row-wise) |
| `axis=1` | Join horizontally (column-wise) |
| `axis=n` | Create/join along the specified dimension |

### Quick Memory Rule

- `concatenate` → General-purpose join
- `hstack` → Horizontal (columns)
- `vstack` → Vertical (rows)
- `column_stack` → Make columns from 1D arrays
- `stack` → Create a new dimension

In [87]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = pd.read_csv('covid_toy.csv')

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [89]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [90]:
df.describe()

,age,fever
count,100.000000,90.000000
mean,44.220000,100.844444
std,24.878931,2.054926
min,5.000000,98.000000
25%,20.000000,99.000000
50%,45.000000,101.000000
75%,66.500000,102.750000
max,84.000000,104.000000


We saw that age,fever belongs to a numerical data and rest are categorical columns.

- Age has no missing values, it is perfect as we can see in `df.describe()` output.
- we will scale fever using simpleImputer & encode `cough`(ordinal data) using ordinal encoder & `city,gender`(nominal data) will be encoded using OHE.
- we can encode `has_covid`(label) using label encoding

In [91]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'], test_size=0.2, random_state=42)
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


SimpleImputer: `fever`

In [92]:
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

OrdinalEncoding: `Cough`

In [93]:
X_train['cough'].value_counts(0)

cough
Mild      48
Strong    32
Name: count, dtype: int64

In [94]:
Oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = Oe.fit_transform(X_train[['cough']])
X_test_cough = Oe.transform(X_test[['cough']])

X_test_cough.shape

(20, 1)

OneHotEncoding: `gender & city`

In [95]:
X_train['city'].value_counts()

city
Bangalore    27
Kolkata      23
Delhi        16
Mumbai       14
Name: count, dtype: int64

In [96]:
OHE = OneHotEncoder(drop='first', sparse_output=False)

X_train_genCity = OHE.fit_transform(X_train[['gender', 'city']])
X_test_genCity = OHE.transform(X_test[['gender', 'city']])

X_train_genCity.shape

(80, 4)

Extracting `age` from the dataset

In [97]:
X_train_age = X_train.drop(columns=['gender','fever','cough', 'city']).values
X_train_age.shape

(80, 1)

Concatinate all the tranformed columns now using numpy concatinate function.

In [98]:
new_train = np.concatenate(
    (X_train_age,
     X_train_genCity,
     X_train_fever,
     X_train_cough),
    axis=1
)

new_train.shape

(80, 7)

## Sklearn  ColumnTransformer

In [99]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(
    transformers=[
            ('odnl',OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
            ('si', SimpleImputer(), ['fever']),
            ('ohe', OneHotEncoder(drop='first', sparse_output= False), ['gender', 'city'])
    ],
    
    remainder='passthrough'
    
)

In [100]:
transformer.fit_transform(X_train).shape

(80, 7)

In [101]:
transformer.fit_transform(X_test).shape

(20, 7)